# L2P for IP102 (Learning to Prompt, CVPR2022)Incremental learning + retrieval (R@1/5/10, mAP) + open-world (AUROC, FPR95) + lifelong (plasticity/forgetting/overall) metrics on the IP102 pest dataset.Dataset: **1 Input duy nhat** (chua `train.json`/`val.json`/`test.json` + thu muc anh `JPEGImages`). Code duoc clone tu GitHub qua env `IP102_CODE_REPO`, neu khong co thi tim trong `/kaggle/input`.

In [ ]:
# ==== 1. Lay code tu GitHub (env IP102_CODE_REPO) hoac /kaggle/input ====import os, sys, glob, subprocessCODE_DIR = Noneif os.environ.get('IP102_CODE_REPO'):    repo = os.environ['IP102_CODE_REPO']    target = '/kaggle/working/L2P-for-IP102'    if not os.path.isdir(target):        print('git clone', repo)        subprocess.run(['git', 'clone', repo, target], check=True)    CODE_DIR = targetelse:    for base in ('/kaggle/input', '/kaggle/working'):        found = sorted(glob.glob(os.path.join(base, '**', 'main_ip102.py'),                                 recursive=True))        if found:            CODE_DIR = os.path.dirname(found[0])            breakif not CODE_DIR:    raise RuntimeError('Khong tim thay code. Dat env IP102_CODE_REPO '                       'hoac day code vao /kaggle/input.')sys.path.insert(0, CODE_DIR)print('CODE_DIR =', CODE_DIR)

In [ ]:
# ==== 2. Cai dat thu vien JAX/Flax (moi nhat, hop python 3.13) ====
import subprocess, sys

# flax>=0.6.0 da go flax.optim (AttributeError) nen repo vendors flax/optim
# (tu flax 0.5.3) va tu dang ky lai qua libml.flax_optim_compat. Vi vay chung
# ta dung jax/flax moi nhat (co wheel cho python 3.13 + cuda12 tren Kaggle).
REQS = ['jax[cuda12]', 'flax', 'clu', 'ml_collections', 'scipy']
if not sys.platform.startswith('linux'):
    REQS = ['jax', 'flax', 'clu', 'ml_collections', 'scipy']
print('pip install', REQS)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] +
               REQS, check=True)

import flax
import jax
from libml.flax_optim_compat import install
install()
from flax import optim  # noqa: F401  (thong qua vendored compat)
import numpy as np
print('flax', flax.__version__, '| jax', jax.__version__,
      '| numpy', np.__version__)
print('flax.optim OK | devices', jax.devices())


In [ ]:
# ==== 3. Kiem tra dataset (auto tim bang deep-walk) ====from libml.ip102_data import get_data_managerdm = get_data_manager('ip102', seed=1993, split_val=True)report = dm.verify()
print('DATA_ROOT =', dm.data_root)
for s in ('train', 'val', 'test'):
    if s not in report:
        continue
    r = report[s]
    print('%-5s images=%-5d anns=%-5d labeled=%-5d missing=%d'
          % (s, r['images_in_json'], r['annotations'],
             r['images_labeled'], r['missing_images']))
assert report['train']['missing_images'] == 0
assert report['num_classes'] == 25
print('num_classes =', report['num_classes'], '| val_split =', report['val_split'])

In [ ]:
# ==== 4. Test nhanh: max_tasks=1 (1 task dau), 1 epoch ====from main_ip102 import run_train

quick = run_train(model='L2P', max_tasks=1, memory_size=0, num_epochs=1)
print('quick results ->', quick)

In [ ]:
# ==== 5. Chay du: max_tasks=0 (toan bo task) ====full = run_train(model='L2P', max_tasks=0, memory_size=0)
print('final results ->', full)

## Ket qua (results.csv)Header: `task,numclass,cnn_top1,nme_top1,R@1,R@5,R@10,mAP,AUROC,FPR95,Plasticity,Forgetting,Overall`(`AUROC/FPR95 = NA` khi da thay du toan bo lop -> khong con OOD de do).

In [ ]:
# ==== 6. Hien thi results.csv (glob dung duong dan noi code chay) ====import glob
import pandas as pd

cands = set()
for base in ('/kaggle/working', CODE_DIR, '.'):
    cands |= set(glob.glob(os.path.join(base, '**', 'results.csv'),
                           recursive=True))
cands = sorted(cands, key=lambda p: os.path.getmtime(p))
print('results.csv files:', cands)
assert cands, 'Khong tim thay results.csv'
path = cands[-1]
print('displaying ->', path)
df = pd.read_csv(path)
display(df)